In this notebook, we will build a 3D map of a scene from a small set of images and then localize an image downloaded from the Internet. This demo was contributed by [Philipp Lindenberger](https://github.com/Phil26AT/).

In [2]:
%load_ext autoreload
%autoreload 2
import tqdm, tqdm.notebook

tqdm.tqdm = tqdm.notebook.tqdm  # notebook-friendly progress bars
from pathlib import Path
import numpy as np

from hloc import (
    extract_features,
    match_features,
    reconstruction,
    visualization,
    pairs_from_exhaustive,
)
from hloc.visualization import plot_images, read_image
from hloc.utils import viz_3d

# Setup
Here we define some output paths.

In [3]:
images = Path("datasets/sacre_coeur")
outputs = Path("outputs/demo")

import shutil

if outputs.exists():
    shutil.rmtree(outputs)

sfm_pairs = outputs / "pairs-sfm.txt"
loc_pairs = outputs / "pairs-loc.txt"
sfm_dir = outputs / "sfm"
features = outputs / "features.h5"
matches = outputs / "matches.h5"

feature_conf = extract_features.confs["aliked-n16"]
matcher_conf = match_features.confs["aliked+lightglue"]

# 3D mapping
First we list the images used for mapping. These are all day-time shots of Sacre Coeur.

In [4]:
references = [
    p.relative_to(images).as_posix()
    for p in (images / "mapping/").iterdir()
]

print(len(references), "mapping images")

10 mapping images


Then we extract features and match them across image pairs. Since we deal with few images, we simply match all pairs exhaustively. For larger scenes, we would use image retrieval, as demonstrated in the other notebooks.

In [5]:
extract_features.main(
    feature_conf, images, image_list=references, feature_path=features
)
pairs_from_exhaustive.main(sfm_pairs, image_list=references)
match_features.main(matcher_conf, sfm_pairs, features=features, matches=matches);

[2026/07/16 16:39:44 hloc INFO] Extracting local features with configuration:
{'model': {'model_name': 'aliked-n16', 'name': 'aliked'},
 'output': 'feats-aliked-n16',
 'preprocessing': {'grayscale': False, 'resize_max': 1024}}


  0%|          | 0/10 [00:00<?, ?it/s]

[2026/07/16 16:39:51 hloc INFO] Finished exporting features.
[2026/07/16 16:39:51 hloc INFO] Found 45 pairs.
[2026/07/16 16:39:51 hloc INFO] Matching local features with configuration:
{'model': {'features': 'aliked', 'name': 'lightglue'},
 'output': 'matches-aliked-lightglue'}


  0%|          | 0/45 [00:00<?, ?it/s]

[2026/07/16 16:39:56 hloc INFO] Finished exporting matches.


The we run incremental Structure-From-Motion and display the reconstructed 3D model.

In [1]:
model = reconstruction.main(
    sfm_dir, images, sfm_pairs, features, matches, image_list=references
)
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(
    fig, model, color="rgba(255,0,0,0.5)", name="mapping", points_rgb=True
)
fig.write_html("mapping.html")

NameError: name 'reconstruction' is not defined

In [2]:
fig = viz_3d.init_figure()

viz_3d.plot_reconstruction(
    fig,
    model,
    color="rgba(255,0,0,0.5)",
    name="mapping",
    points_rgb=True
)

fig.write_html("mapping.html")

NameError: name 'viz_3d' is not defined

We also visualize which keypoints were triangulated into the 3D model.

In [ ]:
import matplotlib.pyplot as plt

plt.savefig(
    "sfm_result.png",
    dpi=300,
    bbox_inches="tight"
)

from pathlib import Path

from hloc.utils import viz_3d
from hloc import reconstruction


In [3]:
sfm_dir = Path("outputs/demo/sfm")

model = reconstruction.load_model(sfm_dir)


NameError: name 'Path' is not defined

# Localization
Now that we have a 3D map of the scene, we can localize any image. To demonstrate this, we download [a night-time image from Wikimedia](https://commons.wikimedia.org/wiki/File:Paris_-_Basilique_du_Sacr%C3%A9_Coeur,_Montmartre_-_panoramio.jpg).

In [ ]:
url = "https://upload.wikimedia.org/wikipedia/commons/5/53/Paris_-_Basilique_du_Sacr%C3%A9_Coeur%2C_Montmartre_-_panoramio.jpg"
# try other queries by uncommenting their url
# url = "https://upload.wikimedia.org/wikipedia/commons/5/59/Basilique_du_Sacr%C3%A9-C%C5%93ur_%285430392880%29.jpg"
# url = "https://upload.wikimedia.org/wikipedia/commons/8/8e/Sacr%C3%A9_C%C5%93ur_at_night%21_%285865355326%29.jpg"
query = "query/night.jpg"
!mkdir -p $images/query && wget $url -O $images/$query -q
plot_images([read_image(images / query)], dpi=75)

Again, we extract features for the query and match them exhaustively.

In [ ]:
extract_features.main(
    feature_conf, images, image_list=[query], feature_path=features, overwrite=True
)
pairs_from_exhaustive.main(loc_pairs, image_list=[query], ref_list=references)
match_features.main(
    matcher_conf, loc_pairs, features=features, matches=matches, overwrite=True
);

We read the EXIF data of the query to infer a rough initial estimate of camera parameters like the focal length. Then we estimate the absolute camera pose using PnP+RANSAC and refine the camera parameters.

In [ ]:
import pycolmap
from hloc.localize_sfm import QueryLocalizer, pose_from_cluster

camera = pycolmap.infer_camera_from_image(images / query)
ref_ids = [model.find_image_with_name(r).image_id for r in references]
conf = {
    "estimation": {"ransac": {"max_error": 12}},
    "refinement": {"refine_focal_length": True, "refine_extra_params": True},
}
localizer = QueryLocalizer(model, conf)
ret, log = pose_from_cluster(localizer, query, camera, ref_ids, features, matches)

print(f'found {ret["num_inliers"]}/{len(ret["inlier_mask"])} inlier correspondences.')
visualization.visualize_loc_from_log(images, query, log, model)

We visualize the correspondences between the query images a few mapping images. We can also visualize the estimated camera pose in the 3D map.

In [ ]:
viz_3d.plot_camera_colmap(
    fig, ret["cam_from_world"], ret["camera"], color="rgba(0,255,0,0.5)", name=query, fill=True,
    text=f"inliers: {ret['num_inliers']} / {ret['inlier_mask'].shape[0]}"
)
# visualize 2D-3D correspodences
inl_3d = np.array(
    [model.points3D[pid].xyz for pid in np.array(log["points3D_ids"])[ret["inlier_mask"]]]
)
viz_3d.plot_points(fig, inl_3d, color="lime", ps=1, name=query)
fig.write_html("mapping.html")


from pathlib import Path

sfm = Path("outputs/demo/sfm")

for f in sfm.iterdir():
    print(f)

In [1]:
from pathlib import Path

sfm = Path("outputs/demo/sfm")

for f in sfm.iterdir():
    print(f)

outputs\demo\sfm\cameras.bin
outputs\demo\sfm\colmap.LOG.20260716-161255.31528
outputs\demo\sfm\database.db
outputs\demo\sfm\frames.bin
outputs\demo\sfm\images.bin
outputs\demo\sfm\models
outputs\demo\sfm\points3D.bin
outputs\demo\sfm\rigs.bin


In [2]:
from pathlib import Path
import csv

import numpy as np
from scipy.spatial.transform import Rotation


pose_file = Path(
    "outputs/my_desk_v2/query_poses.txt"
)

csv_file = Path(
    "outputs/my_desk_v2/query_camera_coordinates.csv"
)

rows = []

for line in pose_file.read_text(
    encoding="utf-8"
).splitlines():

    if not line.strip():
        continue

    values = line.split()

    image_name = values[0]

    # HLoc输出：qw qx qy qz tx ty tz
    qw, qx, qy, qz = map(
        float,
        values[1:5],
    )

    tx, ty, tz = map(
        float,
        values[5:8],
    )

    # SciPy四元数顺序：qx qy qz qw
    rotation = Rotation.from_quat(
        [qx, qy, qz, qw]
    ).as_matrix()

    translation = np.array(
        [tx, ty, tz],
        dtype=float,
    )

    # 相机在COLMAP世界坐标系中的位置
    camera_center = (
        -rotation.T @ translation
    )

    rows.append({
        "image": image_name,
        "X": float(camera_center[0]),
        "Y": float(camera_center[1]),
        "Z": float(camera_center[2]),
        "qw": qw,
        "qx": qx,
        "qy": qy,
        "qz": qz,
        "tx": tx,
        "ty": ty,
        "tz": tz,
    })


print(
    f"{'照片':<12}"
    f"{'X':>14}"
    f"{'Y':>14}"
    f"{'Z':>14}"
)

print("-" * 54)

for row in rows:
    print(
        f"{row['image']:<12}"
        f"{row['X']:>14.6f}"
        f"{row['Y']:>14.6f}"
        f"{row['Z']:>14.6f}"
    )


with csv_file.open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "image",
            "X",
            "Y",
            "Z",
            "qw",
            "qx",
            "qy",
            "qz",
            "tx",
            "ty",
            "tz",
        ],
    )

    writer.writeheader()
    writer.writerows(rows)


print("\n坐标文件已保存：")
print(csv_file.resolve())

照片                       X             Y             Z
------------------------------------------------------
001.jpg           1.279003     -7.280661     10.484770
002.jpg           8.939111     -1.495326     -2.975728
003.jpg           5.321535      1.401876     -2.598444
004.jpg          -3.798622     -5.458800      9.330929
005.jpg           5.884577      0.437109     -8.253546
006.jpg           6.787758      5.166933    -12.449514
007.jpg          -1.718030      0.859852      1.591294
008.jpg          12.901407     -0.913757     -5.615333

坐标文件已保存：
E:\Hierarchical-Localization-master\outputs\my_desk_v2\query_camera_coordinates.csv


In [3]:
from pathlib import Path
import csv

import numpy as np
import pycolmap


# 三维建图模型
sfm_dir = Path(
    "outputs/my_desk_v2/sfm"
)

# 当时视频抽帧图片目录
mapping_dir = Path(
    "datasets/my_desk/mapping"
)

# 坐标输出文件
coordinate_file = Path(
    "outputs/my_desk_v2/mapping_image_coordinates.csv"
)

# 未成功注册图像输出文件
unregistered_file = Path(
    "outputs/my_desk_v2/unregistered_mapping_images.txt"
)


# 加载COLMAP/HLoc三维模型
model = pycolmap.Reconstruction(
    str(sfm_dir)
)


rows = []

# model.images中保存的是成功注册到三维模型的图像
for image_id, image in model.images.items():

    # 相机在世界坐标系中的位置
    center = np.asarray(
        image.projection_center(),
        dtype=float,
    ).reshape(3)

    rows.append({
        "image_id": int(image_id),
        "image": image.name,
        "X": float(center[0]),
        "Y": float(center[1]),
        "Z": float(center[2]),
    })


# 按图片名称排序
rows.sort(
    key=lambda item: item["image"]
)


# 输出到屏幕
print(
    f"{'图片名称':<22}"
    f"{'X':>14}"
    f"{'Y':>14}"
    f"{'Z':>14}"
)

print("-" * 66)

for row in rows:
    print(
        f"{row['image']:<22}"
        f"{row['X']:>14.6f}"
        f"{row['Y']:>14.6f}"
        f"{row['Z']:>14.6f}"
    )


# 保存CSV
with coordinate_file.open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "image_id",
            "image",
            "X",
            "Y",
            "Z",
        ],
    )

    writer.writeheader()
    writer.writerows(rows)


# 查找没有注册成功、因此没有坐标的抽帧图像
image_suffixes = {
    ".jpg",
    ".jpeg",
    ".png",
    ".JPG",
    ".JPEG",
    ".PNG",
}

all_mapping_images = {
    path.relative_to(mapping_dir).as_posix()
    for path in mapping_dir.rglob("*")
    if path.is_file()
    and path.suffix in image_suffixes
}

registered_images = {
    row["image"]
    for row in rows
}

unregistered_images = sorted(
    all_mapping_images - registered_images
)


unregistered_file.write_text(
    "\n".join(unregistered_images),
    encoding="utf-8",
)


print("\n" + "=" * 66)

print("抽帧图片总数：", len(all_mapping_images))
print("有坐标的注册图片：", len(rows))
print("没有坐标的未注册图片：", len(unregistered_images))

print("\n坐标CSV：")
print(coordinate_file.resolve())

print("\n未注册图片名单：")
print(unregistered_file.resolve())

if unregistered_images:
    print("\n未注册图片：")

    for name in unregistered_images:
        print(name)

图片名称                               X             Y             Z
------------------------------------------------------------------
00000.jpg                   6.177649      0.288336      0.530669
00001.jpg                   6.220505      0.278798      0.500581
00002.jpg                   6.295203      0.262090      0.456275
00003.jpg                   6.592359      0.264436      0.311982
00004.jpg                   6.765337      0.189411      0.379640
00005.jpg                   7.098599      0.161280      0.320207
00006.jpg                   7.329458      0.109003      0.367299
00007.jpg                   7.328979      0.068194      0.419368
00008.jpg                   7.401887      0.035036      0.456696
00016.jpg                  -2.320508     -0.445259     -0.638032
00017.jpg                  -2.361069     -0.415467     -1.119521
00018.jpg                  -2.323180     -0.437801     -0.954554
00019.jpg                  -2.259750     -0.472549     -0.662745
00020.jpg              